# 🇱🇰 Sinhala ASR Post-Correction — mBART + LoRA Fine-Tuning

Fine-tunes `facebook/mbart-large-50` with **LoRA (Low-Rank Adaptation)** to correct raw Whisper ASR output for Sinhala.

---

## 🧮 LoRA — The Key Equation

Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times k}$, LoRA decomposes the weight update into two low-rank matrices:

$$W' = W + \frac{\alpha}{r} \cdot B A$$

| Symbol | Meaning |
|--------|---------|
| $W$ | Original frozen pre-trained weight matrix |
| $B \in \mathbb{R}^{d \times r}$ | Low-rank matrix, initialised to **zero** |
| $A \in \mathbb{R}^{r \times k}$ | Low-rank matrix, initialised from $\mathcal{N}(0,\sigma^2)$ |
| $r$ | Rank — controls expressivity vs. parameter budget |
| $\alpha$ | Scaling factor (controls the effective learning rate of the adapter) |
| $\frac{\alpha}{r}$ | Normalisation term so update magnitude is stable across different $r$ choices |

**Why does this work?**
Pre-trained weight matrices have a low intrinsic rank for the adaptation task.  
Only $B$ and $A$ are trained; $W$ is frozen — saving up to ~95 % of trainable parameters compared to full fine-tuning.

---

**What this model learns to fix:**
- ✅ Spelling / phonetic errors  
- ✅ Inverse Text Normalisation (spoken numbers → digits)  
- ✅ Punctuation restoration  
- ✅ Vowel sign restoration  
- ✅ Any combination of the above  

> Hyperparameters from **Optuna HPO study `sinhala_asr_hpo`** — best trial **#89**, best WER = **0.3426**.


## 1. Install Dependencies

In [1]:
!amd-smi

+------------------------------------------------------------------------------+
| AMD-SMI 26.0.0+37d158ab      amdgpu version: 6.14.14  ROCm version: 7.0.0    |
| Platform: Linux Guest                                                        |
|-------------------------------------+----------------------------------------|
| BDF                        GPU-Name | Mem-Uti   Temp   UEC       Power-Usage |
| GPU  HIP-ID  OAM-ID  Partition-Mode | GFX-Uti    Fan               Mem-Usage |
|=====================================+========================================|
| 0000:83:00.0                 0x74b5 | 0 %      41 °C   0           140/750 W |
|   0       0       5        SPX/NPS1 | 0 %        N/A           285/196288 MB |
+-------------------------------------+----------------------------------------+
+------------------------------------------------------------------------------+
| Processes:                                                                   |
|  GPU        PID  Process N

In [2]:
%%capture
!pip install transformers datasets sentencepiece sacrebleu jiwer evaluate accelerate pynvml wandb plotly kaleido -q
!pip install peft -q


## 2. Imports & Reproducibility

In [3]:
import os
import math
import random
import unicodedata
import json
import re
import warnings
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from transformers import (
    MBartForConditionalGeneration,
    MBart50Tokenizer,
    MT5ForConditionalGeneration,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
import evaluate
import wandb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


/opt/venv/lib/python3.10/site-packages/torchao/utils.py:408: UserWarning: TORCH_VERSION_AT_LEAST_2_5 is deprecated and will be removed in torchao 0.14.0
  warnings.warn(self.msg)


✅ Using device: cuda
   GPU: 
   VRAM: 205.8 GB


## 3. ⚙️ Configuration — Model, Dataset & LoRA

> All hyperparameters below are sourced from **Optuna HPO study `sinhala_asr_hpo`**,  
> best trial **#89**, best WER = **0.3426**.  
>  
> LoRA key equation: $W' = W + \dfrac{\alpha}{r} \cdot BA$

#### API KEYS

In [ ]:
HF_TOKEN      = ""
WANDB_API_KEY = ""

if HF_TOKEN:
    from huggingface_hub import login as hf_login
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in to HuggingFace Hub")

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("✅ WandB API key set")


✅ Logged in to HuggingFace Hub
✅ WandB API key set


#### MODEL CONFIG

In [5]:
MODEL_CONFIG = {
    "model_name": "facebook/mbart-large-50",
    "src_lang": "si_LK",
    "tgt_lang": "si_LK",
    "max_input_length": 256,
    "max_target_length": 256,
    "use_mt5_prefix": False,
    "mt5_prefix": "correct sinhala asr: ",
}


#### LoRA CONFIG

The adapter weights satisfy: $W' = W + \dfrac{\alpha}{r} \cdot BA$

| Parameter | Value | Note |
|-----------|-------|------|
| `lora_r` | 121 | Rank $r$ — from Optuna trial #89 |
| `lora_alpha` | 355 | Scaling $\alpha$ — directly from Optuna trial #89 |
| `lora_dropout` | 0.0749 | Dropout on LoRA layers |
| Target modules | `q_proj, v_proj, k_proj, out_proj, fc1, fc2` | `qkvo_fc` — attention + FFN projections |

In [6]:
# ── Optuna HPO study: sinhala_asr_hpo  |  Best trial #89  |  WER = 0.3426 ──
#
# LoRA equation:  W' = W + (alpha / r) * B @ A
#   B  ∈ R^{d × r}  (zero-init)
#   A  ∈ R^{r × k}  (normal-init)
#   r     = lora_r      (rank)
#   alpha = lora_alpha  (given directly by Optuna — not derived from a ratio)
#
# lora_target_modules = "qkvo_fc"
#   → q_proj, k_proj, v_proj, out_proj  (attention)
#   → fc1, fc2                           (feed-forward)

LORA_CONFIG = {
    "lora_r":       121,                        # rank r
    "lora_alpha":   355,                        # scaling α (given directly)
    "lora_dropout": 0.07494014201207418,
    "target_modules": [                         # qkvo_fc
        "q_proj", "k_proj", "v_proj", "out_proj",
        "fc1", "fc2",
    ],
}

print(f"✅ LoRA config  r={LORA_CONFIG['lora_r']}  alpha={LORA_CONFIG['lora_alpha']}  "
      f"dropout={LORA_CONFIG['lora_dropout']:.4f}")
print(f"   Effective scaling α/r = {LORA_CONFIG['lora_alpha'] / LORA_CONFIG['lora_r']:.4f}")
print(f"   Target modules: {LORA_CONFIG['target_modules']}")


✅ LoRA config  r=121  alpha=355  dropout=0.0749
   Effective scaling α/r = 2.9339
   Target modules: ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2']


#### DATASET CONFIG

In [7]:
DATASET_CONFIG = [
    {
        "name": "openslr_sinhala_spell_correction",
        "hf_dataset_name": "SPEAK-PP/openslr-sinhala-spelling-correction-prediction-reference-60000",
        "hf_dataset_config": None,
        "src_col": "dyslexic_sentence",
        "tgt_col": "clean_sentence",
        "enabled": True,
        "apply_noise": False,
        "oversample": 1,
    },
]


#### TRAINING CONFIG — Optuna HPO study `sinhala_asr_hpo`, trial #89, WER = 0.3426

In [ ]:
TRAINING_CONFIG = {
    "output_dir": "./checkpoints/sinhala-asr-correction-lora-t89",

    # ── Optuna HPO values — trial #89 ─────────────────────────────────────
    "num_train_epochs":              20,
    "per_device_train_batch_size":   16,
    "per_device_eval_batch_size":    16,
    "gradient_accumulation_steps":   2,                           # Optuna: 2  → effective batch = 32
    "learning_rate":                 0.00015674516067908906,      # Optuna
    "warmup_steps":                  7,                           # Optuna: direct step count
    "weight_decay":                  0.0009086126884506659,       # Optuna
    "adam_beta1":                    0.9,                         # standard default
    "adam_beta2":                    0.9899051397061436,          # Optuna
    "adam_epsilon":                  4.810818550831622e-09,       # Optuna
    "max_grad_norm":                 2.9286817872909743,          # Optuna
    "lr_scheduler_type":             "linear",                    # Optuna
    "neftune_noise_alpha":           2.001391502192957,           # Optuna — NEFTune regularisation
    "label_smoothing_factor":        0.1481872042251659,          # Optuna
    # ─────────────────────────────────────────────────────────────────────

    "fp16":                          True,
    "eval_strategy":                 "epoch",
    "save_strategy":                 "epoch",
    "load_best_model_at_end":        True,
    "metric_for_best_model":         "wer",
    "greater_is_better":             False,
    "early_stopping_patience":       3,
    "val_split":                     0.05,
    "test_split":                    0.05,
    "use_wandb":                     False,
    "wandb_project":                 "sinhala-asr-correction-lora",
    "generation_num_beams":          4,
    "generation_max_length":         256,
}

print("✅ Training config loaded  (Optuna HPO sinhala_asr_hpo trial #89)")
print(f"   lr={TRAINING_CONFIG['learning_rate']:.6f}  "
      f"wd={TRAINING_CONFIG['weight_decay']:.6f}  "
      f"batch={TRAINING_CONFIG['per_device_train_batch_size']}  "
      f"grad_accum={TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"   scheduler={TRAINING_CONFIG['lr_scheduler_type']}  "
      f"warmup_steps={TRAINING_CONFIG['warmup_steps']}  "
      f"neftune_alpha={TRAINING_CONFIG['neftune_noise_alpha']:.4f}  "
      f"label_smooth={TRAINING_CONFIG['label_smoothing_factor']:.4f}")


✅ Training config loaded  (Optuna HPO sinhala_asr_hpo trial #89)
   lr=0.000157  wd=0.000909  batch=16  grad_accum=2
   scheduler=linear  warmup_steps=7  neftune_alpha=2.0014  label_smooth=0.1482


## 4. Sinhala Text Utilities

In [9]:
def nfc_normalize(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def is_sinhala(text: str) -> bool:
    return any("\u0D80" <= ch <= "\u0DFF" for ch in text)

def clean_text(text: str) -> str:
    text = nfc_normalize(str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def sentence_split(text: str, max_len: int = 200) -> List[str]:
    sentences = re.split(r"(?<=[.!?।෴])\s+", text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) < max_len:
            current += " " + sent
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sent
    if current.strip():
        chunks.append(current.strip())
    return [c for c in chunks if len(c) > 10]

print("✅ Text utilities ready")


✅ Text utilities ready


## 5. Noise Augmentation — Simulating ASR Errors

In [10]:
class SinhalaASRNoiseAugmenter:
    VOWEL_SIGNS = list("ාිීුූෙේෛොෝෞංඃ")
    SIMILAR_CHARS = {
        "ක": ["ඛ", "ග"], "ට": ["ඩ", "ත"], "ප": ["බ", "ඵ"],
        "ස": ["ශ", "ෂ"], "ල": ["ළ"], "ණ": ["න"], "ඩ": ["ද"],
    }
    SPOKEN_NUMBERS = {
        "1": "එක", "2": "දෙක", "3": "තුන", "4": "හතර", "5": "පහ",
        "6": "හය", "7": "හත", "8": "අට", "9": "නව", "10": "දහය",
        "11": "එකොළහ", "12": "දොළහ", "14": "දාහතර", "15": "පහළොව",
        "20": "විස්ස", "22": "දෙවිසි", "100": "සිය", "1000": "දාහ",
    }
    PUNCTUATION = r"[.!?,;:–—\"'()\[\]]"

    def __init__(self, noise_prob: float = 0.3):
        self.noise_prob = noise_prob

    def drop_vowel_signs(self, text):
        return "".join(c for c in text
                       if c not in self.VOWEL_SIGNS or random.random() > self.noise_prob)

    def substitute_similar_chars(self, text):
        chars = list(text)
        for i, ch in enumerate(chars):
            if ch in self.SIMILAR_CHARS and random.random() < self.noise_prob * 0.5:
                chars[i] = random.choice(self.SIMILAR_CHARS[ch])
        return "".join(chars)

    def remove_punctuation(self, text):
        return re.sub(self.PUNCTUATION, "", text).strip()

    def numbers_to_spoken(self, text):
        for digit, spoken in sorted(self.SPOKEN_NUMBERS.items(), key=lambda x: -len(x[0])):
            text = re.sub(r"\b" + re.escape(digit) + r"\b", spoken, text)
        return text

    def add_filler_words(self, text):
        fillers = ["ඇ", "හ්ම්", "ම්", "ඔව්"]
        words = text.split()
        if len(words) > 3 and random.random() < self.noise_prob:
            pos = random.randint(1, len(words) - 1)
            words.insert(pos, random.choice(fillers))
        return " ".join(words)

    def apply(self, text, error_types=None):
        all_types = ["drop_vowel_signs", "substitute_similar_chars",
                     "remove_punctuation", "numbers_to_spoken", "add_filler_words"]
        if error_types is None:
            selected = ["remove_punctuation"]
            selected += [t for t in all_types[:-1] if random.random() < 0.5]
        else:
            selected = error_types
        noisy = text
        for et in selected:
            noisy = getattr(self, et)(noisy)
        return clean_text(noisy)

_aug = SinhalaASRNoiseAugmenter(noise_prob=0.4)
_sample = "ඒ මිනිහා ගෙදර ආවේ 14 වෙනිදා රාත්‍රියේ."
print(f"Original : {_sample}")
print(f"Noisy    : {_aug.apply(_sample)}")
print("✅ Noise augmenter ready")


Original : ඒ මිනිහා ගෙදර ආවේ 14 වෙනිදා රාත්‍රියේ.
Noisy    : ඒ මිනිහා ගෙදර ආවේ දාහතර වෙනිදා රාත්‍රියේ
✅ Noise augmenter ready


## 6. Dataset Loaders

In [11]:
def load_hf_pairs_dataset(cfg: dict, augmenter: SinhalaASRNoiseAugmenter) -> Dataset:
    apply_noise = cfg.get("apply_noise", False)
    raw = load_dataset(cfg["hf_dataset_name"], cfg.get("hf_dataset_config"))
    all_splits = concatenate_datasets([raw[s] for s in raw])
    df = all_splits.to_pandas()
    src_col, tgt_col = cfg["src_col"], cfg["tgt_col"]
    df = df[[src_col, tgt_col]].dropna()
    df["tgt"] = df[tgt_col].apply(clean_text)
    if apply_noise:
        print(f"  ⚡ Injecting ASR noise into '{tgt_col}' column ...")
        df["src"] = df["tgt"].apply(lambda t: augmenter.apply(t))
        df = df[df["src"] != df["tgt"]].reset_index(drop=True)
        print(f"  Generated {len(df)} noisy pairs")
    else:
        df["src"] = df[src_col].apply(clean_text)
        df = df[df["src"] != df["tgt"]].reset_index(drop=True)
        print(f"  Kept {len(df)} pairs after filtering identical src/tgt")
    return Dataset.from_pandas(df[["src", "tgt"]], preserve_index=False)


def load_all_datasets(dataset_configs, augmenter):
    all_datasets = []
    for cfg in dataset_configs:
        if not cfg.get("enabled", True):
            print(f"  ⏭️  Skipping disabled dataset: {cfg['name']}")
            continue
        oversample = cfg.get("oversample", 1)
        noise_flag = "⚡ noise=ON" if cfg.get("apply_noise", False) else "📦 noise=OFF"
        print(f"\n📂 Loading: {cfg['name']}  [{noise_flag}  oversample={oversample}×]")
        ds = load_hf_pairs_dataset(cfg, augmenter)
        print(f"  ✅ {len(ds):,} examples loaded")
        if oversample > 1:
            ds = concatenate_datasets([ds] * oversample)
            print(f"  ⚖️  After oversampling ({oversample}×): {len(ds):,} examples")
        all_datasets.append(ds)
    if not all_datasets:
        raise ValueError("No datasets enabled!")
    combined = concatenate_datasets(all_datasets).shuffle(seed=42)
    print(f"\n🗃️  Total combined examples: {len(combined):,}")
    return combined

print("✅ Dataset loaders ready")


✅ Dataset loaders ready


## 7. Load Datasets

In [12]:
augmenter   = SinhalaASRNoiseAugmenter(noise_prob=0.35)
combined_ds = load_all_datasets(DATASET_CONFIG, augmenter)

val_pct  = TRAINING_CONFIG["val_split"]
test_pct = TRAINING_CONFIG["test_split"]

split1 = combined_ds.train_test_split(test_size=test_pct, seed=42)
split2 = split1["train"].train_test_split(
    test_size=val_pct / (1 - test_pct), seed=42
)

dataset = DatasetDict({
    "train":      split2["train"],
    "validation": split2["test"],
    "test":       split1["test"],
})

print("\n📊 Dataset splits:")
for split, ds in dataset.items():
    print(f"  {split:12s}: {len(ds):>7,} examples")

print("\n📝 Sample pairs:")
for i in range(3):
    ex = dataset["train"][i]
    print(f"  SRC: {ex['src']}")
    print(f"  TGT: {ex['tgt']}")
    print()



📂 Loading: openslr_sinhala_spell_correction  [📦 noise=OFF  oversample=1×]


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.75M [00:00<?, ?B/s]

data/eval-00000-of-00001.parquet:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60417 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/7676 [00:00<?, ? examples/s]

  Kept 62733 pairs after filtering identical src/tgt
  ✅ 62,733 examples loaded

🗃️  Total combined examples: 62,733

📊 Dataset splits:
  train       :  56,459 examples
  validation  :   3,137 examples
  test        :   3,137 examples

📝 Sample pairs:
  SRC: පොඩ්ඩක් වැඩ කරන එක නැවැත්තුවම තමයි ඉහිප දෙනකු බොරු චෝදනා කළා.
  TGT: පොඩ්ඩක් වැඩ කරන එක නැවත්තුවම තමයි. කිහිප දෙනකු බොරු චෝදනා කළා.

  SRC: ඊබට් ජයරත්නව රහස් පොලිසියට කැඳවන ලැබිණි
  TGT: ඊබට්‌ ජයරත්නව රහස්‌ පොලීසියට කැදවනු ලැබිණි

  SRC: අනුවීන් කාලේ ලෝකය පුරාම සිද්ධ වුණ ආර්ථික පසු බැස්න වැල් ඇදගෙන එයි. ඉන්දුස්ථානේ සංගීත ආභාෂයයි.
  TGT: පහුගිය කාලේ ලෝකේ පුරාම සිද්ධ උන ආර්ථික පසුබැස්ම වැල් ඇදගෙන එයි. හින්දුස්ථානි සංගීත ආභාෂයයි.



## 8. Load Base Model & Apply LoRA

The LoRA adapter wraps selected projection layers. Only $B$ and $A$ are trained; the base weights $W$ remain **frozen**.

$$W' = W + \frac{\alpha}{r} \cdot B A$$

In [13]:
model_name = MODEL_CONFIG["model_name"]
print(f"Loading base model: {model_name} ...")

use_mbart = "mbart" in model_name.lower()

if use_mbart:
    tokenizer = MBart50Tokenizer.from_pretrained(
        model_name,
        src_lang=MODEL_CONFIG["src_lang"],
        tgt_lang=MODEL_CONFIG["tgt_lang"],
    )
    base_model = MBartForConditionalGeneration.from_pretrained(model_name)
    forced_bos_token_id = tokenizer.lang_code_to_id[MODEL_CONFIG["tgt_lang"]]
    base_model.config.forced_bos_token_id = None
    base_model.generation_config.forced_bos_token_id = forced_bos_token_id
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = MT5ForConditionalGeneration.from_pretrained(model_name)
    forced_bos_token_id = None

# ── Wrap with LoRA adapters (trial #89 geometry) ─────────────────────────
# W' = W + (alpha / r) * B @ A
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_CONFIG["lora_r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
    bias="none",
)

model = get_peft_model(base_model, lora_cfg)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"\n✅ LoRA model ready  (trial #89)")
print(f"   Total params    : {total_params / 1e6:.1f}M")
print(f"   Trainable (LoRA): {trainable_params / 1e6:.2f}M  "
      f"({100 * trainable_params / total_params:.2f}%)")
print(f"   Frozen (base)   : {frozen_params / 1e6:.1f}M")
model.print_trainable_parameters()


Loading base model: facebook/mbart-large-50 ...


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]


✅ LoRA model ready  (trial #89)
   Total params    : 676.3M
   Trainable (LoRA): 65.42M  (9.67%)
   Frozen (base)   : 610.9M
trainable params: 65,421,312 || all params: 676,300,800 || trainable%: 9.6734


## 9. Tokenization

In [14]:
MAX_IN     = MODEL_CONFIG["max_input_length"]
MAX_TGT    = MODEL_CONFIG["max_target_length"]
USE_PREFIX = MODEL_CONFIG["use_mt5_prefix"]
PREFIX     = MODEL_CONFIG["mt5_prefix"]

def preprocess_function(examples):
    srcs = examples["src"]
    tgts = examples["tgt"]
    if USE_PREFIX:
        srcs = [PREFIX + s for s in srcs]
    model_inputs = tokenizer(srcs, max_length=MAX_IN, truncation=True, padding=False)
    labels = tokenizer(text_target=tgts, max_length=MAX_TGT, truncation=True, padding=False)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets...")
tokenized = dataset.map(
    preprocess_function,
    batched=True, batch_size=1000,
    remove_columns=["src", "tgt"],
    desc="Tokenizing",
)
print("✅ Tokenization complete")
print(f"  Example input_ids length: {len(tokenized['train'][0]['input_ids'])}")


Tokenizing datasets...


Tokenizing:   0%|          | 0/56459 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3137 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3137 [00:00<?, ? examples/s]

✅ Tokenization complete
  Example input_ids length: 21


## 10. Metrics — BLEU + WER + CER

In [15]:
bleu_metric = evaluate.load("sacrebleu")
wer_metric  = evaluate.load("wer")
cer_metric  = evaluate.load("cer")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # When predict_with_generate=True the trainer may return a tuple
    # (generated_ids, scores) — we only need the token ids.
    if isinstance(preds, tuple):
        preds = preds[0]

    # Sanitise predictions: replace -100 padding and clamp to valid vocab range
    # to avoid "piece id is out of range" from the MBart50 sentencepiece model.
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    preds  = np.clip(preds,  0, tokenizer.vocab_size - 1)

    # Sanitise labels the same way
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = bleu_metric.compute(predictions=decoded_preds,
                               references=[[l] for l in decoded_labels])
    wer  = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    cer  = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {
        "bleu": round(bleu["score"], 2),
        "wer":  round(wer, 4),
        "cer":  round(cer, 4),
    }

print("✅ Metrics ready (BLEU + WER + CER)")


✅ Metrics ready (BLEU + WER + CER)


## 11. Training Arguments & Trainer

All values from **Optuna HPO `sinhala_asr_hpo` trial #89** (WER = 0.3426).

In [16]:
if TRAINING_CONFIG["use_wandb"]:
    wandb.init(project=TRAINING_CONFIG["wandb_project"])
    report_to = "wandb"
else:
    os.environ["WANDB_DISABLED"] = "true"
    report_to = "none"

# Trial #89 gives warmup_steps directly (not as a ratio)
_warmup_steps = TRAINING_CONFIG["warmup_steps"]
print(f"   warmup_steps={_warmup_steps}  "
      f"grad_accum={TRAINING_CONFIG['gradient_accumulation_steps']}  "
      f"scheduler={TRAINING_CONFIG['lr_scheduler_type']}")

training_args = Seq2SeqTrainingArguments(
    output_dir=TRAINING_CONFIG["output_dir"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],

    # ── Optuna HPO trial #89 ─────────────────────────────────────────────
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_steps=_warmup_steps,
    weight_decay=TRAINING_CONFIG["weight_decay"],
    adam_beta1=TRAINING_CONFIG["adam_beta1"],
    adam_beta2=TRAINING_CONFIG["adam_beta2"],
    adam_epsilon=TRAINING_CONFIG["adam_epsilon"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    lr_scheduler_type=TRAINING_CONFIG["lr_scheduler_type"],
    neftune_noise_alpha=TRAINING_CONFIG["neftune_noise_alpha"],
    label_smoothing_factor=TRAINING_CONFIG["label_smoothing_factor"],
    # ────────────────────────────────────────────────────────────────────

    fp16=TRAINING_CONFIG["fp16"] and DEVICE == "cuda",
    eval_strategy=TRAINING_CONFIG["eval_strategy"],
    save_strategy=TRAINING_CONFIG["save_strategy"],
    load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
    metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
    greater_is_better=TRAINING_CONFIG["greater_is_better"],
    predict_with_generate=True,
    generation_num_beams=TRAINING_CONFIG["generation_num_beams"],
    generation_max_length=TRAINING_CONFIG["generation_max_length"],
    report_to=report_to,
    logging_steps=50,
    save_total_limit=3,
    dataloader_num_workers=2,
    seed=42,
    data_seed=42,
)

gen_kwargs = {}
if forced_bos_token_id is not None:
    gen_kwargs["forced_bos_token_id"] = forced_bos_token_id
    model.generation_config.forced_bos_token_id = forced_bos_token_id

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=TRAINING_CONFIG["early_stopping_patience"]
    )],
)

print("✅ Trainer ready (LoRA + Optuna HPO trial #89)")


   warmup_steps=7  grad_accum=2  scheduler=linear
✅ Trainer ready (LoRA + Optuna HPO trial #89)


## 12. Train

In [17]:
print("🚀 Starting LoRA fine-tuning (trial #89)...")
train_result = trainer.train()

output_dir = TRAINING_CONFIG["output_dir"]
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
print(f"\n✅ Training complete. LoRA adapter saved to: {output_dir}")


🚀 Starting LoRA fine-tuning (trial #89)...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Wer,Cer
1,3.145100,3.100240,48.090000,0.339800,0.093000
2,3.008400,3.022778,49.990000,0.325600,0.089900
3,2.904700,2.992910,50.690000,0.313900,0.087900
4,2.813300,2.973938,51.550000,0.310000,0.087300
5,2.736100,2.966292,52.900000,0.299900,0.085000
6,2.674800,2.969388,53.000000,0.298000,0.082800
7,2.616400,2.962653,54.230000,0.291800,0.083600
8,2.568200,2.968807,54.580000,0.290700,0.083100
9,2.528600,2.983011,55.110000,0.288300,0.084200
10,2.507300,2.991941,55.250000,0.286100,0.082800


***** train metrics *****
  epoch                    =       10.0
  total_flos               = 56557444GF
  train_loss               =     2.7687
  train_runtime            = 1:32:44.46
  train_samples_per_second =    101.463
  train_steps_per_second   =      3.172

✅ Training complete. LoRA adapter saved to: ./checkpoints/sinhala-asr-correction-lora-t89


## 13. 📈 Training Curves

In [18]:
log_history = trainer.state.log_history

train_loss_log = [(e["epoch"], e["loss"]) for e in log_history if "loss" in e and "eval_loss" not in e]
eval_rows      = [e for e in log_history if "eval_loss" in e]

if eval_rows:
    epochs    = [e["epoch"]     for e in eval_rows]
    eval_loss = [e["eval_loss"] for e in eval_rows]
    bleu_vals = [e.get("eval_bleu", None) for e in eval_rows]
    wer_vals  = [e.get("eval_wer",  None) for e in eval_rows]
    cer_vals  = [e.get("eval_cer",  None) for e in eval_rows]

    fig_loss = go.Figure()
    if train_loss_log:
        t_ep, t_loss = zip(*train_loss_log)
        fig_loss.add_trace(go.Scatter(x=t_ep, y=t_loss, mode="lines",
                                      name="Train Loss", line=dict(color="#EF553B")))
    fig_loss.add_trace(go.Scatter(x=epochs, y=eval_loss, mode="lines+markers",
                                  name="Val Loss", line=dict(color="#636EFA")))
    fig_loss.update_layout(title="Training & Validation Loss — LoRA trial #89",
                            xaxis_title="Epoch", yaxis_title="Loss",
                            template="plotly_dark", height=400)
    fig_loss.show()

    fig_metrics = make_subplots(rows=1, cols=3,
                                subplot_titles=["BLEU ↑", "WER ↓", "CER ↓"])
    colors = ["#00CC96", "#FFA15A", "#AB63FA"]
    for col_idx, (vals, label, color) in enumerate(
        zip([bleu_vals, wer_vals, cer_vals], ["BLEU", "WER", "CER"], colors), 1
    ):
        if any(v is not None for v in vals):
            fig_metrics.add_trace(
                go.Scatter(x=epochs, y=vals, mode="lines+markers",
                           name=label, line=dict(color=color)),
                row=1, col=col_idx,
            )
    fig_metrics.update_layout(title="Validation Metrics per Epoch — LoRA trial #89",
                               template="plotly_dark", height=400, showlegend=False)
    fig_metrics.show()
else:
    print("ℹ️  No eval metrics found — run training first.")


## 14. Evaluate on Test Set

In [19]:
print("📊 Evaluating on test set...")
test_results = trainer.evaluate(tokenized["test"], metric_key_prefix="test")

print("\n🏆 Test Results:")
for k, v in test_results.items():
    print(f"  {k}: {v}")

trainer.save_metrics("test", test_results)


📊 Evaluating on test set...


early stopping required metric_for_best_model, but did not find eval_wer so early stopping is disabled



🏆 Test Results:
  test_loss: 2.990885019302368
  test_bleu: 55.26
  test_wer: 0.2856
  test_cer: 0.0832
  test_runtime: 147.1402
  test_samples_per_second: 21.32
  test_steps_per_second: 1.339
  epoch: 10.0


## 15. Inference Pipeline

The LoRA adapter is merged back into the base weights at inference time using `merge_and_unload()`, which collapses $W' = W + \frac{\alpha}{r} BA$ into a single weight matrix — zero extra latency.

In [20]:
class SinhalaASRCorrectorLoRA:
    """Inference wrapper — merges LoRA adapter into base weights for zero-latency generation."""

    def __init__(self, adapter_path: str, device: str = DEVICE):
        print(f"Loading LoRA adapter from: {adapter_path}")
        self.device = device
        if use_mbart:
            base = MBartForConditionalGeneration.from_pretrained(model_name)
            self.tokenizer = MBart50Tokenizer.from_pretrained(
                adapter_path,
                src_lang=MODEL_CONFIG["src_lang"],
                tgt_lang=MODEL_CONFIG["tgt_lang"],
            )
        else:
            base = MT5ForConditionalGeneration.from_pretrained(model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(adapter_path)
        peft_model = PeftModel.from_pretrained(base, adapter_path)
        self.model = peft_model.merge_and_unload().to(device).eval()
        print("✅ LoRA adapter merged — corrector ready")

    @torch.no_grad()
    def correct(self, texts: List[str], num_beams: int = 4) -> List[str]:
        if MODEL_CONFIG["use_mt5_prefix"]:
            texts = [MODEL_CONFIG["mt5_prefix"] + t for t in texts]
        inputs = self.tokenizer(
            texts, return_tensors="pt", max_length=MAX_IN,
            truncation=True, padding=True,
        ).to(self.device)
        gen_kw = dict(num_beams=num_beams, max_length=MAX_TGT, early_stopping=True)
        if use_mbart:
            gen_kw["forced_bos_token_id"] = (
                self.tokenizer.lang_code_to_id[MODEL_CONFIG["tgt_lang"]]
            )
        outputs = self.model.generate(**inputs, **gen_kw)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)


corrector = SinhalaASRCorrectorLoRA(TRAINING_CONFIG["output_dir"])

test_inputs = [
    "මම ගෙදර යනව",
    "ඒ මිනිහ ගෙදර ආව දාහතර වෙනිද රෑ",
    "ඔහු හ්ම් ඒ ස්ථානයට ගිය",
]

corrections = corrector.correct(test_inputs)
print("\n🔍 Inference Demo:")
for src, tgt in zip(test_inputs, corrections):
    print(f"  ASR  : {src}")
    print(f"  Fixed: {tgt}")
    print()


Loading LoRA adapter from: ./checkpoints/sinhala-asr-correction-lora-t89
✅ LoRA adapter merged — corrector ready

🔍 Inference Demo:
  ASR  : මම ගෙදර යනව
  Fixed: මම ගෙදර යනව

  ASR  : ඒ මිනිහ ගෙදර ආව දාහතර වෙනිද රෑ
  Fixed: ඒ මිනිහ ගෙදර ආව දාහතරවෙනිද රෑ

  ASR  : ඔහු හ්ම් ඒ ස්ථානයට ගිය
  Fixed: ඔහු හ්ම් ඒ ස්ථානයට ගිය



## 16. Push to HuggingFace Hub (Optional)

In [21]:
HF_REPO = "SPEAK-PP/sinhala-asr-corrector-lora-v2"   # ← change this

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"✅ LoRA adapter pushed to: https://huggingface.co/{HF_REPO}")


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...p_acgvbdb/adapter_model.safetensors:   0%|          | 1.11MB /  262MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...tmpabdv1nwt/sentencepiece.bpe.model:  94%|#########3| 4.74MB / 5.07MB            

✅ LoRA adapter pushed to: https://huggingface.co/SPEAK-PP/sinhala-asr-corrector-lora-t89
